# SOXX 2019-06 原始公开 NPORT-EX Part F 来源

SEC 同一 series 的30只普通股原件；仅在申报后的真实Qlib交易日生效，不读收益或运行账户。

In [1]:
import os,sys,json,hashlib,re,datetime,tarfile,tempfile,subprocess,shlex
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd,qlib
from qlib.data import D
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import _settings,nas_api_call
card=root/'research/stage1_soxx_2019_june_original_part_f_source_20261023';snap=root/'data/snapshots/stage1_soxx_2019_june_original_part_f_source_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['source'];receipt=json.loads((card/'transport_receipts.json').read_text());state=json.loads((card/'run_state.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and state['new_HTTP_GET_attempts']==2 and receipt['GETs_consumed']==2 and all(x['HTTP_status']==200 for x in receipt['attempts'])
assert sha(root/src['calendar_provider']/'qlib.yaml')==src['calendar_qlib_yaml_sha256']
paths={kind:snap/(kind+'.html.response') for kind in ('index','body')}
for item in receipt['attempts']:
 assert item['URL']==src[item['kind']+'_url'] and sha(paths[item['kind']])==item['body_sha256'] and paths[item['kind']].stat().st_size==item['body_bytes']
index=paths['index'].read_bytes();body=paths['body'].read_bytes();state['raw_source_reads']=2;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
ix=BeautifulSoup(index,'html.parser');it=' '.join(ix.stripped_strings)
for key in ('accession','report_period','filed_date','accepted_ET','form','series','class'):
 assert str(src[key]) in it,(key,src[key])
assert 'SOXX' in it and 'NPORT_8256748838153481.htm' in it
soup=BeautifulSoup(body,'html.parser');bt=' '.join(soup.stripped_strings)
assert 'iShares PHLX Semiconductor ETF' in bt and 'June 30, 2019' in bt
rows=[];inside=False;printed=None
for tr in soup.find_all('tr'):
 vals=[' '.join(td.stripped_strings) for td in tr.find_all(['td','th'],recursive=False)];line=' '.join(vals)
 if 'Common Stocks' in line and not inside:inside=True;continue
 if inside and 'Total Common Stocks' in line:
  printed=int(vals[7].replace(',',''));break
 if not inside or len(vals)<8:continue
 name=vals[0].strip();sh=vals[3].replace(',','').strip();value=vals[7].replace(',','').strip()
 if name and re.fullmatch(r'[0-9]+',sh) and re.fullmatch(r'[0-9]+',value):
  rows.append({'security_original':name,'shares_original':int(sh),'fair_value_usd_original':int(value)})
assert len(rows)==30 and len({x['security_original'] for x in rows})==30 and all(x['shares_original']>0 and x['fair_value_usd_original']>0 for x in rows)
assert printed==1473254424 and sum(x['fair_value_usd_original'] for x in rows)==printed
for x in rows:x['reported_equity_weight']=x['fair_value_usd_original']/printed
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['calendar_provider']),region='us',kernels=1)
cal=pd.DatetimeIndex(D.calendar(start_time='2019-08-20',end_time='2019-09-03',freq='day'));filed=pd.Timestamp(src['filed_date']);effective=cal[int(cal.searchsorted(filed,side='right'))];assert effective==pd.Timestamp('2019-08-27') and effective>filed
state['D_calendar_calls']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
result={'decision':'original_SOXX_2019Jun_public_PartF_30_equity_source_qualified','plan_sha256':sha(card/'plan.json'),'accession':src['accession'],'form':src['form'],'SEC_CIK':src['SEC_CIK'],'series':src['series'],'class':src['class'],'report_period':src['report_period'],'filed_date':src['filed_date'],'accepted_ET':src['accepted_ET'],'first_conservative_Qlib_signal_date':str(effective.date()),'index_sha256':sha(paths['index']),'body_sha256':sha(paths['body']),'original_equity_rows':len(rows),'printed_common_stocks_total_usd':printed,'parsed_common_stocks_total_usd':sum(x['fair_value_usd_original'] for x in rows),'total_closure_usd':0,'equities':rows,'SOXX_market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
settings=_settings(root);files=[card/'plan.json',card/'transport_receipts.json',card/'result.json',paths['index'],paths['body']]
with tempfile.TemporaryDirectory(prefix='soxx-jun2019-partf-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for p in files:out.add(p,arcname=str(p.relative_to(root)))
 digest=sha(archive);response=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research');uploaded=response['artifact'];assert uploaded['sha256']==digest and uploaded['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;restored=Path(tmp)/'restored.tar.gz'
 with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(restored)==digest
 with tarfile.open(restored,'r:gz') as z:
  names={m.name:m for m in z.getmembers() if m.isfile()};assert set(names)=={str(p.relative_to(root)) for p in files}
  for p in files:assert hashlib.sha256(z.extractfile(names[str(p.relative_to(root))]).read()).hexdigest()==sha(p)
archive_receipt={'sha256':digest,'size_bytes':uploaded['size_bytes'],'nas_path':remote,'members':{str(p.relative_to(root)):sha(p) for p in files},'remote_and_scoped_restore_verified':True};(card/'nas_source_archive.json').write_text(json.dumps(archive_receipt,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_archive':'nas_source_archive.json','report_period':src['report_period'],'first_conservative_Qlib_signal_date':str(effective.date()),'source_NAS_sha256':digest,'SOXX_market_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Public Part F first becomes available after SEC filing, not on June report date.','This 30-equity source closes one period gap but does not certify full 2011-2026 legal issuer identity or turnover factor.','No market outcome, fit or funded Stage1 account.']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['phase']='finished';state['NAS_sha256']=digest;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print({'accession':result['accession'],'equity_rows':len(rows),'total':printed,'effective':str(effective.date()),'NAS':digest})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[50940:MainThread](2026-10-23 10:00:24,895) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[50940:MainThread](2026-10-23 10:00:25,197) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[50940:MainThread](2026-10-23 10:00:25,198) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


{'accession': '0001752724-19-108721', 'equity_rows': 30, 'total': 1473254424, 'effective': '2019-08-27', 'NAS': '0531795250d4def59d7263f861ca083725642b0b057b41cdb40f3cc1e696af72'}


# SOXX 2019-06 原始公开 Part F：过渡期披露断点已补

[SEC 原始 NPORT-EX 索引](https://www.sec.gov/Archives/edgar/data/1100663/0001752724-19-108721-index.htm)明确标记 SOXX series `S000004354`、class `C000012084`，报告期2019-06-30、申报/受理为2019-08-26 13:39:47 ET。对应[原文](https://www.sec.gov/Archives/edgar/data/1100663/000175272419108721/NPORT_8256748838153481.htm)列出30只普通股及各自股数、市值；逐行合计 **$1,473,254,424**，与印刷普通股总额美元精确闭合，未纳入两只货币基金或短期投资。原始索引和正文 SHA、每行原值见 `result.json`。

按冻结源时钟，报告期6月末**不回填**。受理后的第一个真实 Qlib 交易信号日为 **2019-08-27**；8月26日09:46前不能使用该文件。[SEC 2019 N-PORT 过渡 FAQ](https://www.sec.gov/rules-regulations/staff-guidance/investment-company-reporting-modernization-frequently-asked-questions)说明3–8月的 N-PORT 主体非公开，但部分 Part F 投资组合附件公开，解释了先前只看 NPORT-P 造成的断点。首次公开分钟仍以本份 SEC 原始受理钟作保守边界。

这份公开原件的两次 GET、原字节、传输收据、30行提取结果已归档 Yanxuan NAS，远端 SHA-256 与限定恢复通过。**决定：2019-06 SOXX 原始篮子来源资格通过**，可另行核对2019-03→06与06→09证券身份；尚不等于2011–2026全期换手因子、预测增量或0bp费用账户。本卡0未来 SOXX 收益读取、0拟合、0账户。
